# 06. So sánh và dự đoán email mới

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/ml_scratch').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Mở notebook trong dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import display, Image


## Mục đích
mục đích: Trên cùng 977 email test, KNN và Naive Bayes khác nhau chỗ nào, và email mới đi qua những file nào?

Input: `artifacts/knn.json`, `artifacts/naive_bayes.json`, `outputs/knn_predictions.csv`, `outputs/naive_bayes_predictions.csv`. `compare()` kiểm tra hai file cùng `source_row` và cùng nhãn thật.

Spam là lớp dương. Accuracy dễ cao vì ham chiếm khoảng 70%. Precision và FPR nói về ham bị chặn nhầm. Recall nói về spam bị sót. Demo hai thư bên dưới không nằm trong test và không cộng vào các chỉ số này.

In [ ]:
from ml_scratch.evaluation import compare, demo
import pandas as pd
compare()
demo()
display(pd.read_csv(ROOT/"outputs/knn_confusion_matrix.csv", index_col=0))
display(pd.read_csv(ROOT/"outputs/naive_bayes_confusion_matrix.csv", index_col=0))
display(pd.read_csv(ROOT/"outputs/reports/demo.csv"))

## Hai hình so sánh
`evaluation_comparison.png` đặt accuracy, precision, recall, F1 và AUC của KNN và Naive Bayes cạnh nhau. Các số lấy từ lần chấm trên cùng 977 email.

`demo.png` chỉ vẽ spam score của thư họp và thư giải thưởng. Hai thư này không nằm trong test và không cộng vào các chỉ số trên.

In [ ]:
display(Image(filename=str(ROOT/"outputs/figures/evaluation_comparison.png")))
display(Image(filename=str(ROOT/"outputs/figures/demo.png")))

## Email mới đi qua những bước nào?
Nạp JSON → `clean_text` → `vectorizer.transform` (không fit) → `model.predict` → nhãn. Từ ngoài từ vựng bị bỏ. Email không còn từ nào trong vocabulary thành vector 0: KNN vẫn bỏ phiếu theo hàng xóm của vector 0, Naive Bayes chỉ còn log prior.

Điểm spam của KNN là tỷ lệ phiếu. Điểm của Naive Bayes là thành phần spam sau `normalized_scores`. Cả hai chưa calibration.

Demo đã lưu: thư họp ra ham (KNN score 0; Naive Bayes khoảng 0,11), thư giải thưởng ra spam (KNN score 1; Naive Bayes khoảng 0,93). `demo.png` chỉ vẽ hai thư này. Chúng không có trong 977 dòng test.

In [ ]:
from ml_scratch.json_io import load_json
from ml_scratch.data import clean_text
bundle = load_json(ROOT/"artifacts/naive_bayes.json")
email = "Subject: Please review the team report tomorrow"
x = bundle["vectorizer"].transform([clean_text(email)])
print(bundle["model"].predict(x))

## Kết quả so sánh

Output: `outputs/reports/model_comparison.csv`, `demo.csv`, `evaluation_comparison.png`, `demo.png`.

Cùng 685 ham và 292 spam. KNN FP 20 / FN 15, FPR 2,92%, recall 94,86%. Naive Bayes FP 53 / FN 18, FPR 7,74%, recall 93,84%. Trên split này KNN ít chặn nhầm ham hơn và cũng sót ít spam hơn. Một lần chia, chưa đo tốc độ, chưa có tập ngoài: không kết luận thuật toán nào luôn tốt hơn.